# date_time

In [ ]:
///- --test static

open testing

In [ ]:
open rust.rust_operators
open sm'_operators

## date_time

### timestamp

In [ ]:
nominal timestamp_gleam =
    `(
        backend_switch `(()) `({}) {
            Gleam = (fun () => global "import gleam/time/timestamp") : () -> ()
        }
        $'' : $'timestamp.Timestamp'
    )
nominal timestamp = i64

inl timestamp forall t {int}. (x : t) : timestamp =
    timestamp (convert.convert x : i64)

inl timestamp_gleam_of ((timestamp x) : timestamp) : timestamp_gleam =
    global "import gleam/time/timestamp"
    $'timestamp.from_unix_seconds_and_nanoseconds(!x / 1000000, !x % 1000000 * 1000)'

### timestamp_guid

In [ ]:
type timestamp_guid = guid.guid

### date_time_guid

In [ ]:
type date_time_guid = guid.guid

### test_guid

In [ ]:
///- --test static

inl test_guid () =
    "6543210F-EDCB-A987-6543-210FEDCBA987" |> guid.new_guid

## day_of_week

In [ ]:
union day_of_week =
    | Sunday
    | Monday
    | Tuesday
    | Wednesday
    | Thursday
    | Friday
    | Saturday

## month

In [ ]:
union month =
    | January
    | February
    | March
    | April
    | May
    | June
    | July
    | August
    | September
    | October
    | November
    | December

## day

In [ ]:
nominal day = int

## year

In [ ]:
nominal year = int

## fsharp

### date_time

In [ ]:
nominal date_gleam =
    `(
        backend_switch `(()) `({}) {
            Gleam = (fun () =>
                global "import gleam/time/calendar"
                        ) : () -> ()
        }
        $'' : $'calendar.Date'
    )
nominal time_of_day_gleam =
    `(
        backend_switch `(()) `({}) {
            Gleam = (fun () =>
                global "import gleam/time/calendar"
                        ) : () -> ()
        }
        $'' : $'calendar.TimeOfDay'
    )
nominal duration_gleam =
    `(
        backend_switch `(()) `({}) {
            Gleam = (fun () =>
                global "import gleam/time/duration"
                        ) : () -> ()
        }
        $'' : $'duration.Duration'
    )

nominal date_time_python =
    `(
        backend_switch `(()) `({}) {
            Python = (fun () => global "import datetime") : () -> ()
        }
        $'' : $'datetime.datetime'
    )
type date_time_switch =
    {
        Gleam : date_gleam * time_of_day_gleam * duration_gleam
        Lua : i64
        Fsharp : $'System.DateTime'
        Python : date_time_python
        Rust : $'chrono::NaiveDateTime'
        TypeScript : $'Date'
    }
nominal date_time = $'backend_switch `(date_time_switch)'

### calendar_month

In [ ]:
nominal calendar_month =
    `(
        backend_switch `(()) `({}) {
            Gleam = (fun () =>
                global "import gleam/time/calendar"
                        ) : () -> ()
        }
        $'' : $'calendar.Month'
    )

### tempo_date_time

In [ ]:
nominal tempo_date_time =
    `(
        backend_switch `(()) `({}) {
            Gleam = (fun () =>
                global "import tempo"
                        ) : () -> ()
        }
        $'' : $'tempo.DateTime'
    )

### year

In [ ]:
inl get_year (date_time : date_time) : i32 =
    backend_switch {
        Gleam = fun () => $'0' : i32
        Lua = fun () => $'0' : i32
        Fsharp = fun () => date_time |> $'_.Year' : i32
        Python = fun () => $'!date_time.year' : i32
        Rust = fun () => $'chrono::Datelike::year(&!date_time)' : i32
    }

### split_date_time

In [ ]:
inl split_date_time (date_time : date_time) : date_gleam * time_of_day_gleam * duration_gleam =
    $'!date_time.0', $'!date_time.1', $'!date_time.2'

### new_timestamp_gleam

In [ ]:
inl new_timestamp_gleam
    (date, time_of_day, duration : date_gleam * time_of_day_gleam * duration_gleam)
    : timestamp_gleam
    =
    global "import gleam/time/timestamp"
    inl result : timestamp_gleam = $'timestamp.from_calendar(!date, !time_of_day, !duration)'

    global "import tempo/datetime"
    inl tempo_date_time : tempo_date_time =
        result |> $'datetime.from_timestamp'
    inl tempo_date_time : tempo_date_time = $'!tempo_date_time |> datetime.add(!duration)'
    tempo_date_time |> $'datetime.to_timestamp'

### format

In [ ]:
inl format (format : string) (date_time : date_time) : string =
    backend_switch {
        Gleam = fun () =>
            global "import tempo/datetime"
            inl tempo_date_time : tempo_date_time =
                date_time |> split_date_time |> new_timestamp_gleam |> $'datetime.from_timestamp'
            inl format =
                if format = ""
                then "M-D-YY hh:mm:ss A"
                else format
            $'!tempo_date_time |> datetime.format(tempo.Custom(!format))' : string
        Lua = fun () =>
            inl date_time = join date_time
            if format <> ""
            then $'os.date(!format, !date_time)' : string
            else
                inl date =
                    if get_year date_time < 1000
                    then $'(os.date("%m-%d-%y", !date_time) or ""):gsub("0(%d)", "%1")' : string
                    else $'(os.date("%m-%d-%Y", !date_time) or ""):gsub("0(%d)", "%1")' : string
                inl time =
                    $'os.date("%I:%M:%S %p", !date_time) or ""' : string
                date ++# " " ++# time
        Fsharp = fun () =>
            inl format =
                if format = ""
                then "M-d-y hh:mm:ss tt"
                else format
            $'!date_time.ToString' format : string
        Python = fun () =>
            inl date_time = join date_time
            if format <> ""
            then $'!date_time.strftime(!format)' : string
            elif get_year date_time < 1000
            then $'\'{dt.month}-{dt.day}-{dt.year} {dt:%I}:{dt:%M}:{dt:%S} {dt:%p}\'.format(dt=!date_time)' : string
            else $'\'{dt.month}-{dt.day}-{dt:%y} {dt:%I}:{dt:%M}:{dt:%S} {dt:%p}\'.format(dt=!date_time)' : string
        Rust = fun () =>
            inl format =
                if format = ""
                then "%-m-%-d-%-y %I:%M:%S %p"
                else format
            $'Rc::<str>::from(!date_time.format(&*!format).to_string())' : string
    }

### format_iso8601

In [ ]:
inl format_iso8601 (date_time : date_time) : string =
    backend_switch {
        Gleam = fun () => date_time |> format "YYYY-MM-DDTHH-mm-ss.SSS" : string
        Lua = fun () => date_time |> format "%Y-%m-%dT%H-%M-%S.%f" : string
        Fsharp = fun () => date_time |> format "yyyy-MM-ddTHH-mm-ss.fff" : string
        Python = fun () => date_time |> format "%Y-%m-%dT%H-%M-%S.%f" : string
        Rust = fun () => date_time |> format "%Y-%m-%dT%H-%M-%S.%3f" : string
    }

### min_value

In [ ]:
inl min_value () : date_time =
    backend_switch {
        Gleam = fun () =>
            inl date : date_gleam = $'calendar.Date(1, calendar.January, 1)' : date_gleam
            inl time_of_day : time_of_day_gleam = $'calendar.TimeOfDay(0, 0, 0, 0)' : time_of_day_gleam
            inl duration : duration_gleam = $'duration.seconds(0) ' : duration_gleam
            (date, time_of_day, duration) |> convert.to : date_time
        Lua = fun () =>
            inl date_time = $'os.time{year=1, month=1, day=1, hour=0, min=0, sec=0} or -62135596800' : date_time
            date_time
        Fsharp = fun () => $'System.DateTime.MinValue' : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime.min' : date_time
        Rust = fun () => $'chrono::NaiveDate::from_ymd_opt(1, 1, 1).unwrap().and_hms_opt(0, 0, 0).unwrap()' : date_time
    }

In [1]:
///- --test
///> fsharp
///> cuda

min_value ()
|> format ""
|> _assert_eq "1-1-1 12:00:00 AM"

.fsx (fsharp)
__assert_eq { expected = 1-1-1 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-1 12:00:00 AM }

### max_value

In [ ]:
inl max_value () : date_time =
    backend_switch {
        Gleam = fun () =>
            inl date : date_gleam = $'calendar.Date(9999, calendar.December, 31)' : date_gleam
            inl time_of_day : time_of_day_gleam = $'calendar.TimeOfDay(23, 59, 59, 999)' : time_of_day_gleam
            inl duration : duration_gleam = $'duration.seconds(0)  ' : duration_gleam
            (date, time_of_day, duration) |> convert.to : date_time
        Lua = fun () =>
            inl date_time = $'os.time{year=9999, month=12, day=31, hour=23, min=59, sec=59} or 253402300799' : date_time
            date_time
        Fsharp = fun () => $'System.DateTime.MaxValue' : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime.max' : date_time
        Rust = fun () => $'chrono::NaiveDate::from_ymd_opt(9999, 12, 31).unwrap().and_hms_nano_opt(23, 59, 59, 999999900).unwrap()' : date_time
    }

In [2]:
///- --test
///> fsharp
///> cuda

max_value ()
|> format ""
|> _assert_eq "12-31-99 11:59:59 PM"

.fsx (fsharp)
__assert_eq { expected = 12-31-99 11:59:59 PM }
.py (cuda)
__assert_eq { expected = 12-31-99 11:59:59 PM }

### to_calendar_utc

In [ ]:
inl to_calendar_utc_gleam (timestamp : timestamp_gleam) : date_time =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/time/calendar"
            global "import gleam/time/timestamp"
    }
    inl duration = $'calendar.utc_offset' : duration_gleam
    inl (date : date_gleam), (time_of_day : time_of_day_gleam) = $'!timestamp |> timestamp.to_calendar(!duration)'
    (date, time_of_day, duration) |> convert.to

inl to_calendar_utc ((timestamp micros) as timestamp : timestamp) : date_time =
    backend_switch {
        Gleam = fun () => timestamp |> timestamp_gleam_of |> to_calendar_utc_gleam
        Lua = fun () => $'math.floor(!micros / 1000000)' : date_time
        Fsharp = fun () => $'System.DateTime.UnixEpoch.AddTicks(!micros * 10L)' : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime(1970, 1, 1) + datetime.timedelta(microseconds=!micros)' : date_time
        Rust = fun () => $'chrono::DateTime::from_timestamp_micros(!micros).unwrap().naive_utc()' : date_time
    }

### tempo_date_time

In [ ]:
nominal time_zone_provider =
    `(
        backend_switch `(()) `({}) {
            Gleam = (fun () =>
                global "import tempo"
                        ) : () -> ()
        }
        $'' : $'tempo.TimeZoneProvider'
    )

### unix_epoch

In [ ]:
inl unix_epoch () : date_time =
    backend_switch {
        Gleam = fun () =>
            0i32 |> timestamp |> to_calendar_utc : date_time
        Lua = fun () =>
            inl date_time = $'os.time{year=1970, month=1, day=1, hour=0, min=0, sec=0} or 0' : date_time
            date_time
        Fsharp = fun () => $'System.DateTime.UnixEpoch' : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime(1970, 1, 1)' : date_time
        Rust = fun () => $'chrono::NaiveDate::from_ymd_opt(1970, 1, 1).unwrap().and_hms_opt(0, 0, 0).unwrap()' : date_time
    }

In [3]:
///- --test
///> lua
///> fsharp
///> gleam
///> cuda

unix_epoch ()
|> format ""
|> _assert_eq "1-1-70 12:00:00 AM"

.lua (lua)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.fsx (fsharp)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.gleam (gleam)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-70 12:00:00 AM }

### date_time_milliseconds

In [ ]:
inl date_time_milliseconds
    (year : int) (month : int) (day : int) (hour : int) (minute : int) (second : int) (millisecond : int)
    : date_time
    =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/time/calendar"
            global "import gleam/time/duration"
            global "import gleam/time/timestamp"
            global "import tempo/month"
            inl calendar_month : resultm.result' calendar_month () =
                $"!month |> month.from_int"
            inl calendar_month = calendar_month |> resultm.unbox |> resultm.get
            inl date : date_gleam = $'calendar.Date(#year, #calendar_month, #day)'
            inl nanoseconds = millisecond * 1000000
            inl time_of_day : time_of_day_gleam = $'calendar.TimeOfDay(#hour, #minute, #second, #nanoseconds)'
            inl duration : duration_gleam = $'duration.milliseconds(#millisecond)'
            (date, time_of_day, duration) |> convert.to : date_time
        Lua = fun () =>
            inl total_seconds = second + (minute * 60) + (hour * 3600)
            inl date_time = $'os.time{year=!year, month=!month, day=!day, hour=0, min=0, sec=0} or 0' : int
            date_time + (total_seconds + (millisecond / 1000)) |> convert.unbox : date_time
        Fsharp = fun () =>
            $'System.DateTime (!year, !month, !day, !hour, !minute, !second, !millisecond)' : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime(!year, !month, !day, !hour, !minute, !second, !millisecond * 1000)' : date_time
        Rust = fun () =>
            $'chrono::NaiveDate::from_ymd_opt(!year, !month as u32, !day as u32).unwrap().and_hms_milli_opt(!hour as u32, !minute as u32, !second as u32, !millisecond as u32).unwrap()' : date_time
        TypeScript = fun () => $'new Date(!year, !month - 1, !day, !hour, !minute, !second, !millisecond)' : date_time
    }

In [4]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda

date_time_milliseconds 1970 1 1 0 0 0 0
|> format ""
|> _assert_eq "1-1-70 12:00:00 AM"

.gleam (gleam)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.lua (lua)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.fsx (fsharp)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-70 12:00:00 AM }

### date_time_utc

In [ ]:
inl date_time_utc
    (year : int) (month : int) (day : int) (hour : int) (minute : int) (second : int)
    : date_time
    =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/time/calendar"
            global "import gleam/time/duration"
            global "import gleam/time/timestamp"
            global "import tempo/month"
            inl calendar_month : resultm.result' calendar_month () =
                $"!month |> month.from_int"
            inl calendar_month = calendar_month |> resultm.unbox |> resultm.get
            inl date : date_gleam = $'calendar.Date(#year, #calendar_month, #day)'
            inl time_of_day : time_of_day_gleam = $'calendar.TimeOfDay(#hour, #minute, #second, 0)'
            inl duration : duration_gleam = $'duration.seconds(0)   ' : duration_gleam
            (date, time_of_day, duration) |> convert.to : date_time
        Lua = fun () =>
            inl total_seconds = second + (minute * 60) + (hour * 3600)
            inl date_time = $'os.time{year=!year, month=!month, day=!day, hour=0, min=0, sec=0} or 0' : int
            date_time + total_seconds |> convert.unbox : date_time
        Fsharp = fun () =>
            $'System.DateTime (!year, !month, !day, !hour, !minute, !second, System.DateTimeKind.Utc)' : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime(!year, !month, !day, !hour, !minute, !second, tzinfo=datetime.timezone.utc)' : date_time
        Rust = fun () =>
            $'chrono::NaiveDate::from_ymd_opt(!year, !month as u32, !day as u32).unwrap().and_hms_opt(!hour as u32, !minute as u32, !second as u32).unwrap()' : date_time
    }

In [5]:
///- --test
///> lua
///> fsharp
///> cuda

date_time_utc 1970 1 1 0 0 0
|> format ""
|> _assert_eq "1-1-70 12:00:00 AM"

.lua (lua)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.fsx (fsharp)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-70 12:00:00 AM }

### date_time_kind

In [ ]:
union date_time_kind =
    | Unspecified
    | Utc
    | Local

### specify_date_kind

In [ ]:
inl specify_date_kind (kind : date_time_kind) (date_time : date_time) : date_time =
    backend_switch {
        Gleam = fun () => date_time
        Lua = fun () => date_time
        Fsharp = fun () =>
            inl kind : $'System.DateTimeKind' =
                match kind with
                | Unspecified => $'System.DateTimeKind.Unspecified'
                | Utc => $'System.DateTimeKind.Utc'
                | Local => $'System.DateTimeKind.Local'
            $'System.DateTime.SpecifyKind (!date_time, !kind)' : date_time
        Python = fun () => $'!date_time.replace(tzinfo=None)' : date_time
        Rust = fun () => date_time
    }

### to_universal_time

In [ ]:
inl to_universal_time (date_time : date_time) : date_time =
    backend_switch {
        Gleam = fun () => date_time
        Lua = fun () => date_time
        Fsharp = fun () => date_time |> $'_.ToUniversalTime()' : date_time
        Python = fun () =>
            global "import datetime"
            $'!date_time.replace(tzinfo=datetime.timezone.utc)' : date_time
        Rust = fun () => date_time
    }

In [6]:
///- --test
///> lua
///> fsharp
///> cuda

date_time_milliseconds 1970 1 1 0 0 0 0
|> specify_date_kind Utc
|> to_universal_time
|> format ""
|> _assert_eq "1-1-70 12:00:00 AM"

.lua (lua)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.fsx (fsharp)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-70 12:00:00 AM }

In [7]:
///- --test
///> lua
///> fsharp
///> cuda

date_time_utc 1970 1 1 0 0 0
|> specify_date_kind Utc
|> format ""
|> _assert_eq "1-1-70 12:00:00 AM"

.lua (lua)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.fsx (fsharp)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-70 12:00:00 AM }

In [8]:
///- --test
///> lua
///> fsharp
///> cuda

date_time_utc 1970 1 1 0 0 0
|> specify_date_kind Local
|> format ""
|> _assert_eq "1-1-70 12:00:00 AM"

.lua (lua)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.fsx (fsharp)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-70 12:00:00 AM }

In [9]:
///- --test
///> lua
///> fsharp
///> cuda

date_time_utc 1970 1 1 0 0 0
|> specify_date_kind Unspecified
|> format ""
|> _assert_eq "1-1-70 12:00:00 AM"

.lua (lua)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.fsx (fsharp)
__assert_eq { expected = 1-1-70 12:00:00 AM }
.py (cuda)
__assert_eq { expected = 1-1-70 12:00:00 AM }

### time_span

In [ ]:
nominal time_span_python =
    `(
        backend_switch `(()) `({}) {
            Python = (fun () => global "import datetime") : () -> ()
        }
        $'' : $'datetime.timedelta'
    )
type time_span_switch =
    {
        Gleam : $'duration.Duration'
        Lua : $'number'
        Fsharp : $'System.TimeSpan'
        Python : time_span_python
        Rust : $'chrono::TimeDelta'
        TypeScript : $'number'
    }
nominal time_span = $'backend_switch `(time_span_switch)'

inl time_span x : time_span =
    backend_switch {
        Gleam = fun () => x |> convert.unbox : time_span
        Lua = fun () => x |> convert.unbox : time_span
        Fsharp = fun () => $'!x |> `time_span ' : time_span
        Python = fun () =>
            global "import datetime"
            $'datetime.timedelta(!x)' : time_span
        Rust = fun () =>
            $'(chrono::TimeDelta::microseconds((!x as i64) / 10) + chrono::TimeDelta::nanoseconds(((!x as i64) % 10) * 100))' : time_span
    }

### new_time_span

In [ ]:
inl new_time_span (a : date_time) (b : date_time) : time_span =
    run_target function
        | Gleam => fun () =>
            inl timestamp : timestamp_gleam = a |> split_date_time |> new_timestamp_gleam
            inl timestamp' : timestamp_gleam = b |> split_date_time |> new_timestamp_gleam
            inl duration : duration_gleam = $'!timestamp |> timestamp.difference(!timestamp')'
            ($'duration.to_seconds(!duration)' : float)
            |> convert.convert
            |> fun x => x : int
            |> convert.convert
        | _ => fun () => $'!b - !a '

### total_seconds

In [ ]:
inl total_seconds (time_span : time_span) : f64 =
    inl n : i64 =
        run_target function
            | Gleam => fun () =>
                backend_switch {
                    Gleam = fun () => global "import gleam/time/duration"
                }
                inl duration : duration_gleam = time_span |> $'duration.seconds'
                duration
                |> $'duration.to_seconds'
                |> (*) 10000000f64
                |> convert.convert
            | Lua => fun () =>
                inl seconds = time_span |> convert.unbox : f64
                seconds * 10000000 |> convert.convert : i64
            | Cuda => fun () => $'!time_span.total_seconds() * 10000000' : i64
            | Rust => fun () =>
                $'{ let t = !time_span; t.num_seconds() * 10000000 + (t - chrono::TimeDelta::seconds(t.num_seconds())).num_nanoseconds().unwrap() / 100 }' : i64
            | _ => fun () => time_span |> $'_.Ticks' : i64
    n / 10000000 |> convert.convert

### days

In [ ]:
inl days (time_span : time_span) : f64 =
    backend_switch {
        Gleam = fun () =>
            backend_switch {
                Gleam = fun () => global "import gleam/time/duration"
            }
            inl duration : duration_gleam = time_span |> $'duration.seconds'
            duration
            |> $'duration.to_seconds'
            |> (/) 86400f64
        Lua = fun () =>
            inl seconds = time_span |> convert.unbox : f64
            seconds / 86400 |> convert.convert : f64
        Fsharp = fun () => time_span |> $'_.TotalDays' : f64
        Python = fun () => $'!time_span.days' : f64
        Rust = fun () =>
            $'{ let t = !time_span; (t.num_seconds() * 10000000 + (t - chrono::TimeDelta::seconds(t.num_seconds())).num_nanoseconds().unwrap() / 100) as f64 / 864000000000.0 }' : f64
    }

### to_timestamp

In [ ]:
inl to_timestamp (date_time : date_time) : timestamp =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/time/timestamp"
            inl t : timestamp_gleam = date_time |> split_date_time |> new_timestamp_gleam
            ($'timestamp.to_unix_seconds_and_nanoseconds(!t).0 * 1000000 + timestamp.to_unix_seconds_and_nanoseconds(!t).1 / 1000' : i64) |> timestamp
        Lua = fun () => ($'!date_time * 1000000' : i64) |> timestamp
        Fsharp = fun () => ($'(!date_time.Ticks - 621355968000000000L) / 10L' : i64) |> timestamp
        Python = fun () =>
            global "import datetime"
            ($'(!date_time.replace(tzinfo=None) - datetime.datetime(1970, 1, 1)) // datetime.timedelta(microseconds=1)' : i64) |> timestamp
        Rust = fun () => ($'!date_time.and_utc().timestamp_micros()' : i64) |> timestamp
    }

In [10]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust -d chrono

unix_epoch ()
|> to_timestamp
|> _assert_eq' (0i64 |> timestamp)

date_time_utc 2026 10 13 12 0 0
|> to_timestamp
|> _assert_eq' (1791892800000000i64 |> timestamp)

.gleam (gleam)
__assert_eq' { expected = 0 }
__assert_eq' { expected = 1791892800000000 }
.fsx (fsharp)
__assert_eq' { expected = 0 }
__assert_eq' { expected = 1791892800000000 }
.py (cuda)
__assert_eq' { expected = 0 }
__assert_eq' { expected = 1791892800000000 }
.rs (rust)
__assert_eq' { expected = 0 }
__assert_eq' { expected = 1791892800000000 }

#### to_timestamp_range (test)

In [11]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust -d chrono

min_value ()
|> to_timestamp
|> _assert_eq' (-62135596800000000i64 |> timestamp)

.gleam (gleam)
__assert_eq' { expected = -62135596800000000 }
.fsx (fsharp)
__assert_eq' { expected = -62135596800000000 }
.py (cuda)
__assert_eq' { expected = -62135596800000000 }
.rs (rust)
__assert_eq' { expected = -62135596800000000 }

#### timestamp_round_trip (test)

In [12]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust -d chrono

1791892800123456i64
|> timestamp
|> to_calendar_utc
|> to_timestamp
|> _assert_eq' (1791892800123456i64 |> timestamp)

.gleam (gleam)
__assert_eq' { expected = 1791892800123456 }
.fsx (fsharp)
__assert_eq' { expected = 1791892800123456 }
.py (cuda)
__assert_eq' { expected = 1791892800123456 }
.rs (rust)
__assert_eq' { expected = 1791892800123456 }

### time_span_of_micros

In [ ]:
inl time_span_of_micros (micros : i64) : time_span =
    backend_switch {
        Gleam = fun () => micros / 1000000 |> convert.unbox : time_span
        Lua = fun () => $'!micros / 1000000' : time_span
        Fsharp = fun () => $'System.TimeSpan.FromTicks(!micros * 10L)' : time_span
        Python = fun () =>
            global "import datetime"
            $'datetime.timedelta(microseconds=!micros)' : time_span
        Rust = fun () => $'chrono::TimeDelta::microseconds(!micros)' : time_span
    }

### hours

In [ ]:
inl hours (time_span : time_span) : i32 =
    backend_switch {
        Gleam = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            ((total_seconds / 3600.0) |> math.floor |> convert.unbox : i32) % 24
        Lua = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            ((total_seconds / 3600.0) |> math.floor |> convert.unbox : i32) % 24
        Fsharp = fun () => time_span |> $'_.Hours' : i32
        Python = fun () => $'!time_span.seconds // 3600' : i32
        Rust = fun () => $'(!time_span.num_hours() % 24) as i32' : i32
        TypeScript = fun () => $'(Math.trunc(!time_span / 3600000) % 24)' : i32
    }

In [13]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust -d chrono
///> typescript

(join date_time_milliseconds 2002 2 2 20 22 24 26)
|> new_time_span (date_time_milliseconds 2001 1 1 10 11 12 13)
|> hours
|> _assert_eq 10

.gleam (gleam)
__assert_eq { expected = 10 }
.lua (lua)
__assert_eq { expected = 10 }
.fsx (fsharp)
__assert_eq { expected = 10 }
.py (cuda)
__assert_eq { expected = 10 }
.rs (rust)
__assert_eq { expected = 10 }
.ts (typescript)
__assert_eq { expected = 10 }

### minutes

In [ ]:
inl minutes (time_span : time_span) : i32 =
    backend_switch {
        Gleam = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            ((total_seconds / 60.0) |> math.floor |> convert.unbox : i32) % 60
        Lua = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            ((total_seconds / 60.0) |> math.floor |> convert.unbox : i32) % 60
        Fsharp = fun () => time_span |> $'_.Minutes' : i32
        Python = fun () => $'(!time_span.seconds // 60) % 60' : i32
        Rust = fun () => $'(!time_span.num_minutes() % 60) as i32' : i32
        TypeScript = fun () => $'(Math.trunc(!time_span / 60000) % 60)' : i32
    }

In [14]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust -d chrono
///> typescript

(join date_time_milliseconds 2002 2 2 20 22 24 26)
|> new_time_span (date_time_milliseconds 2001 1 1 10 11 12 13)
|> minutes
|> _assert_eq 11

.gleam (gleam)
__assert_eq { expected = 11 }
.lua (lua)
__assert_eq { expected = 11 }
.fsx (fsharp)
__assert_eq { expected = 11 }
.py (cuda)
__assert_eq { expected = 11 }
.rs (rust)
__assert_eq { expected = 11 }
.ts (typescript)
__assert_eq { expected = 11 }

### seconds

In [ ]:
inl seconds (time_span : time_span) : i32 =
    backend_switch {
        Gleam = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            (total_seconds |> math.floor |> convert.unbox : i32) % 60
        Lua = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            (total_seconds |> math.floor |> convert.unbox : i32) % 60
        Fsharp = fun () => time_span |> $'_.Seconds' : i32
        Python = fun () => $'!time_span.seconds % 60' : i32
        Rust = fun () => $'(!time_span.num_seconds() % 60) as i32' : i32
        TypeScript = fun () => $'(Math.trunc(!time_span / 1000) % 60)' : i32
    }

In [15]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust -d chrono
///> typescript

(join date_time_milliseconds 2002 2 2 20 22 24 26)
|> new_time_span (date_time_milliseconds 2001 1 1 10 11 12 13)
|> seconds
|> _assert_eq 12

.gleam (gleam)
__assert_eq { expected = 12 }
.lua (lua)
__assert_eq { expected = 12 }
.fsx (fsharp)
__assert_eq { expected = 12 }
.py (cuda)
__assert_eq { expected = 12 }
.rs (rust)
__assert_eq { expected = 12 }
.ts (typescript)
__assert_eq { expected = 12 }

### milliseconds

In [ ]:
inl milliseconds (time_span : time_span) : i32 =
    backend_switch {
        Gleam = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            ((total_seconds * 1000.0) |> math.floor |> convert.unbox : i32) % 1000
        Lua = fun () =>
            inl total_seconds = (time_span |> convert.to : i32) |> convert.convert : f64
            ((total_seconds * 1000.0) |> math.floor |> convert.unbox : i32) % 1000
        Fsharp = fun () => time_span |> $'_.Milliseconds' : i32
        Python = fun () => $'!time_span.microseconds // 1000' : i32
        Rust = fun () => $'(!time_span.num_milliseconds() % 1000) as i32' : i32
        TypeScript = fun () => $'(Math.trunc(!time_span) % 1000)' : i32
    }

In [16]:
///- --test
///> fsharp
///> cuda
///> rust -d chrono
///> typescript

(join date_time_milliseconds 2002 2 2 20 22 24 26)
|> new_time_span (date_time_milliseconds 2001 1 1 10 11 12 13)
|> milliseconds
|> _assert_eq 13

.fsx (fsharp)
__assert_eq { expected = 13 }
.py (cuda)
__assert_eq { expected = 13 }
.rs (rust)
__assert_eq { expected = 13 }
.ts (typescript)
__assert_eq { expected = 13 }

### time_span_format

In [ ]:
inl time_span_format (format : string) (time_span : time_span) : string =
    run_target_args (fun () => format, time_span) function
        | TypeScript => fun format, time_span =>
            $'(() => { const t = !time_span; const f = !format; const neg = t < 0; const a = Math.abs(t); const d = Math.floor(a / 86400000); const h = Math.floor(a / 3600000) % 24; const m = Math.floor(a / 60000) % 60; const s = Math.floor(a / 1000) % 60; const ms = Math.round(a % 1000); const p = (n: number) => String(n).padStart(2, "0"); if (f === "" || f === "c") return (neg ? "-" : "") + (d > 0 ? d + "." : "") + p(h) + ":" + p(m) + ":" + p(s) + (ms > 0 ? "." + String(ms * 10000).padStart(7, "0") : ""); return f.replace("hh", p(h)).replace("mm", p(m)).replace("ss", p(s)); })()' : string
        | Rust => fun format, time_span =>
            $'{ let t = !time_span; let f: &str = &(!format); let neg = t < chrono::TimeDelta::zero(); let a = if neg { -t } else { t }; let n = a.num_seconds() * 10000000 + (a - chrono::TimeDelta::seconds(a.num_seconds())).num_nanoseconds().unwrap() / 100; let (d, h, m, s, x) = (n / 864000000000, (n / 36000000000) % 24, (n / 600000000) % 60, (n / 10000000) % 60, n % 10000000); if f.is_empty() || f == "c" { Rc::<str>::from(format\!("{}{}{:02}:{:02}:{:02}{}", if neg { "-" } else { "" }, if d > 0 { format\!("{}.", d) } else { std::string::String::new() }, h, m, s, if x > 0 { format\!(".{:07}", x) } else { std::string::String::new() })) } else { Rc::<str>::from(f.replace("hh", &format\!("{:02}", h)).replace("mm", &format\!("{:02}", m)).replace("ss", &format\!("{:02}", s))) } }' : string
        | _ => fun format, time_span =>
            backend_switch {
                Gleam = fun () =>
                    global "import tempo/datetime"
                    inl tempo_date_time : tempo_date_time =
                        time_span
                        |> $'duration.seconds'
                        |> fun x => x : duration_gleam
                        |> $'datetime.from_duration'
                    inl format =
                        if format = ""
                        then "c"
                        else format
                    $'!tempo_date_time |> datetime.format(tempo.Custom(!format))' : string
                Lua = fun () =>
                    inl total_seconds = time_span |> convert.unbox : f64
                    inl hours = total_seconds / 3600
                    inl minutes = (total_seconds % 3600) / 60
                    inl seconds = total_seconds % 60
                    inl result =
                        sm'.pad_left 2 '0' #?hours
                        ++# ":"
                        ++# sm'.pad_left 2 '0' #?minutes
                        ++# ":"
                        ++# sm'.pad_left 2 '0' #?seconds
                    inl ms = time_span |> milliseconds
                    if ms <= 0
                    then result
                    else
                        inl ms =
                            #?**ms
                            |> sm'.pad_left 3 '0'
                            |> sm'.pad_right 7 '0'
                        result ++# "." ++# ms
                Fsharp = fun () => $'!time_span.ToString !format ' : string
                Python = fun () =>
                    inl date = $'!time_span + !(min_value ())' : date_time
                    inl time = $'!date.strftime("%H:%M:%S")' : string
                    inl days = time_span |> days
                    inl result =
                        if days <= 0
                        then time
                        else #?**days ++# "." ++# time
                    inl ms = time_span |> milliseconds
                    if ms <= 0
                    then result
                    else
                        inl ms =
                            #?**ms
                            |> sm'.pad_left 3 '0'
                            |> sm'.pad_right 7 '0'
                        result ++# "." ++# ms
            }

In [17]:
///- --test
///> fsharp
///> cuda
///> rust -d chrono
///> typescript

(join date_time_milliseconds 2002 2 2 20 22 24 26)
|> new_time_span (date_time_milliseconds 2001 1 1 10 11 12 13)
|> time_span_format "c"
|> _assert_eq "397.10:11:12.0130000"

.fsx (fsharp)
__assert_eq { expected = 397.10:11:12.0130000 }
.py (cuda)
__assert_eq { expected = 397.10:11:12.0130000 }
.rs (rust)
__assert_eq { expected = 397.10:11:12.0130000 }
.ts (typescript)
__assert_eq { expected = 397.10:11:12.0130000 }

In [18]:
///- --test
///> fsharp
///> cuda
///> rust -d chrono
///> typescript

(join date_time_milliseconds 2002 2 2 20 22 24 26)
|> new_time_span (date_time_milliseconds 2002 2 2 10 11 12 26)
|> time_span_format "c"
|> _assert_eq "10:11:12"

.fsx (fsharp)
__assert_eq { expected = 10:11:12 }
.py (cuda)
__assert_eq { expected = 10:11:12 }
.rs (rust)
__assert_eq { expected = 10:11:12 }
.ts (typescript)
__assert_eq { expected = 10:11:12 }

In [19]:
///- --test
///> fsharp
///> cuda
///> rust -d chrono
///> typescript

(join date_time_milliseconds 2002 2 2 20 22 24 999)
|> new_time_span (date_time_milliseconds 2002 2 2 10 11 12 0)
|> time_span_format "c"
|> _assert_eq "10:11:12.9990000"

.fsx (fsharp)
__assert_eq { expected = 10:11:12.9990000 }
.py (cuda)
__assert_eq { expected = 10:11:12.9990000 }
.rs (rust)
__assert_eq { expected = 10:11:12.9990000 }
.ts (typescript)
__assert_eq { expected = 10:11:12.9990000 }

### time_zone_info

In [ ]:
type time_zone_info_switch =
    {
        Gleam : $'System.TimeZoneInfo'
        Lua : $'System.TimeZoneInfo'
        Fsharp : $'System.TimeZoneInfo'
        Python : $'System.TimeZoneInfo'
        Rust : $'chrono::FixedOffset'
    }
nominal time_zone_info = $'backend_switch `(time_zone_info_switch)'

### add_days

In [ ]:
inl add_days (days : i32) (date_time : date_time) : date_time =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/time/duration"
            inl duration : duration_gleam = $'duration.days(#days)' : duration_gleam
            inl (date : date_gleam), (time_of_day : time_of_day_gleam), (existing_duration : duration_gleam) =
                date_time |> split_date_time
            inl new_duration : duration_gleam = $'!existing_duration + !duration '
            (date, time_of_day, new_duration) |> convert.to : date_time
        Lua = fun () =>
            inl seconds = days * 86400
            (date_time |> convert.unbox) + seconds |> convert.unbox : date_time
        Fsharp = fun () =>
            $'!date_time.AddDays' days : date_time
        Python = fun () =>
            global "import datetime"
            $'!date_time + datetime.timedelta(days=!days)' : date_time
        Rust = fun () => $'!date_time + chrono::TimeDelta::days(!days as i64)' : date_time
    }

### to_calendar

In [ ]:
inl to_calendar_gleam (timestamp : timestamp_gleam) : date_time =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/time/timestamp"
            global "import gleam/time/calendar"
    }
    inl duration = $'calendar.local_offset()' : duration_gleam
    inl (date : date_gleam), (time_of_day : time_of_day_gleam) = $'!timestamp |> timestamp.to_calendar(!duration)'
    (date, time_of_day, duration) |> convert.to

inl to_calendar (timestamp : timestamp) : date_time =
    timestamp |> timestamp_gleam_of |> to_calendar_gleam

### now

In [ ]:
inl now () : date_time =
    backend_switch {
        Gleam = fun () =>
            inl timestamp = $'timestamp.system_time ()' : timestamp_gleam
            timestamp |> to_calendar_gleam
        Lua = fun () =>
            inl date_time = $'os.time()' : date_time
            date_time
        Fsharp = fun () =>
            run_target function
                | _ => fun () => $'System.DateTime.Now'
            : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime.now()' : date_time
        Rust = fun () => $'chrono::Local::now().naive_local()' : date_time
    }

### utc_now

In [ ]:
inl utc_now () : date_time =
    backend_switch {
        Gleam = fun () =>
            inl timestamp = $'timestamp.system_time ()' : timestamp_gleam
            timestamp |> to_calendar_utc_gleam
        Lua = fun () =>
            inl date_time = $'os.time()' : date_time
            date_time
        Fsharp = fun () => $'System.DateTime.UtcNow' : date_time
        Python = fun () =>
            global "import datetime"
            $'datetime.datetime.utcnow()' : date_time
        Rust = fun () => $'chrono::Utc::now().naive_utc()' : date_time
    }

### stopwatch

In [ ]:
nominal stopwatch_python =
    `(
        global "import timeit"
        $'' : $'timeit.default_timer'
    )
type stopwatch_switch =
    {
        Fsharp : $'System.Diagnostics.Stopwatch'
        Python : stopwatch_python
        Rust : $'std::time::Instant'
    }
nominal stopwatch = $'backend_switch `(stopwatch_switch)'

inl stopwatch () : stopwatch =
    backend_switch {
        Fsharp = fun () => $'`stopwatch ()' : stopwatch
        Python = fun () => $'`stopwatch ' : stopwatch
        Rust = fun () => $'std::time::Instant::now()' : stopwatch
    }

inl stopwatch_elapsed_milliseconds (stopwatch : stopwatch) : i64 =
    if is_rust ()
    then $'(!stopwatch.elapsed().as_millis() as i64)'
    else $'!stopwatch.ElapsedMilliseconds'

inl stopwatch_start (stopwatch : stopwatch) : () =
    if is_rust () |> not
    then $'!stopwatch.Start' ()

## rust

### duration'

In [ ]:
nominal duration' =
    `(
        rust_type
            `($'std_time_Duration')
            `($'std::time::Duration')
            "type std_time_Duration = class end"
    )

### date_time'

In [ ]:
nominal date_time' t =
    `(
        rust_type
            `($'chrono_DateTime<`t>')
            `($'chrono::DateTime<`t>')
            "type chrono_DateTime<'T> = class end"
    )

### local

In [ ]:
nominal local =
    `(
        rust_type
            `($'chrono_Local')
            `($'chrono::Local')
            "type chrono_Local = class end"
    )

### naive_date_time

In [ ]:
nominal naive_date_time =
    `(
        rust_type
            `($'chrono_NaiveDateTime')
            `($'chrono::NaiveDateTime')
            "type chrono_NaiveDateTime = class end"
    )

## utc

In [ ]:
nominal utc =
    `(
        rust_type
            `($'chrono_Utc')
            `($'chrono::Utc')
            "type chrono_Utc = class end"
    )

### naive_utc

In [ ]:
inl naive_utc (date_time : date_time' utc) : naive_date_time =
    !\\(date_time, $'"$0.naive_utc()"')

### to_local

In [ ]:
inl to_local (date_time : date_time' utc) : date_time' local =
    inl naive_date_time = date_time |> naive_utc
    !\\(naive_date_time, $'"chrono::offset::TimeZone::from_utc_datetime(&chrono::Local, &$0)"')

### from_timestamp_micros

In [ ]:
inl from_timestamp_micros forall t {number; int}. (timestamp : t) : option (date_time' utc) =
    inl result : optionm'.option' (date_time' utc) =
        !\\(timestamp, $'"chrono::DateTime::from_timestamp_micros($0)"')
    result |> optionm'.unbox

### format'

In [ ]:
inl format' (format : string) (date_time : date_time' utc) : sm'.std_string =
    !\\((date_time, #format), $'"$0.format($1).to_string()"')

### format''

In [ ]:
inl format'' (format : string) (date_time : date_time' _) : sm'.std_string =
    !\\((date_time, #format), $'"$0.format($1).to_string()"')

### format_timestamp

In [ ]:
inl format_timestamp forall t {number; int}. (timestamp : t) =
    inl timestamp = join timestamp
    (timestamp / 1000)
    |> from_timestamp_micros
    |> optionm.map fun x =>
        x
        |> to_local
        |> format'' "%Y-%m-%d %H:%M:%S"
        |> sm'.from_std_string
    |> resultm.from_option

### duration_from_millis

In [ ]:
inl duration_from_millis (ms : u64) : duration' =
    inl ms = join ms
    !\($'"std::time::Duration::from_millis(!ms)"')

## date_time

### time_zone_local

In [ ]:
inl time_zone_local () : time_zone_info =
    run_target function
        | Fsharp _ => fun () =>
            $'System.TimeZoneInfo.Local'
        | Rust => fun () => $'*chrono::Local::now().offset()' : time_zone_info
        | _ => fun () => null ()

### get_utc_offset

In [ ]:
inl get_utc_offset (time_zone_info : time_zone_info) (date_time : date_time) : time_span =
    run_target function
        | Fsharp _ => fun () => date_time |> $'_.GetUtcOffset' (time_zone_local ())
        | Rust => fun () =>
            $'chrono::TimeDelta::seconds(chrono::TimeZone::offset_from_local_datetime(&chrono::Local, &!date_time).earliest().map(|o| o.local_minus_utc()).unwrap_or(0) as i64)' : time_span
        | target => fun () =>
            backend_switch {
                Gleam = fun () =>
                    global "import gleam/time/duration"
                    inl duration : duration_gleam = $'calendar.utc_offset()' : duration_gleam
                    duration
                    |> $'duration.to_seconds'
                    |> ((*) 10000000i32)
                    |> convert.convert
                    |> fun x => x : int
                    |> convert.unbox
                    : time_span
                Lua = fun () =>
                    inl date_time = date_time |> convert.unbox : int
                    inl offset = 5i32 * 3600i32
                    offset * 10000000 |> convert.convert : time_span
                Fsharp = fun () => failwith $'$"date_time.get_utc_offset / target: {!target}"' : time_span
                Python = fun () => $'!date_time.utcoffset()' : time_span
            }

### date_time_guid_from_date_time

In [ ]:
let date_time_guid_from_date_time (guid' : guid.guid) (date_time : date_time) =
    inl create prefix time_zone : date_time_guid =
        inl guid_range =
            guid'
            |> sm'.obj_to_string
            |> sm'.range
                (am'.Start (##prefix + ##time_zone))
                (am'.End eval)
        prefix ++# time_zone ++# guid_range |> guid.new_guid
    run_target function
        | Rust => fun () =>
            inl prefix : string =
                $'{ let s = !date_time.format("%Y%m%d-%H%M-%S%f").to_string(); Rc::<str>::from(format\!("{}-{}-{}", &s[0..18], &s[18..22], &s[22..23])) }'
            inl time_zone = date_time |> get_utc_offset (time_zone_local ())
            inl time_zone_signal = if hours time_zone > 0 then 1u8 else 0
            inl time_zone_value = time_zone |> time_span_format (join "hhmm")
            inl time_zone = #?time_zone_signal ++# time_zone_value
            create prefix time_zone
        | target => fun () =>
            inl prefix = date_time |> format (join "yyyyMMdd-HHmm-ssff-ffff-f")
            inl time_zone = date_time |> get_utc_offset (time_zone_local ())
            inl time_zone_signal = if hours time_zone > 0 then 1u8 else 0
            inl time_zone_value = time_zone |> time_span_format (join "hhmm")
            inl time_zone = #?time_zone_signal ++# time_zone_value
            create prefix time_zone

In [20]:
///- --test
///> fsharp
///> rust -d chrono

inl suffix = test_guid () |> sm'.obj_to_string |> sm'.range (am'.End fun x => x () - 6i32) (am'.End eval)
unix_epoch ()
|> specify_date_kind Utc
|> to_universal_time
|> date_time_guid_from_date_time (test_guid ())
|> sm'.obj_to_string
|> fun s => s |> _assert_eq' ((s |> sm'.slice 0i32 29) ++# suffix)

.fsx (fsharp)
__assert_eq' { expected = 19700101-0000-0000-0000-000000cba987 }
.rs (rust)
__assert_eq' { expected = 19700101-0000-0000-0000-000000cba987 }

In [21]:
///- --test --no-agree "local UTC offset in year 1 is implementation-defined"
///> fsharp
///> rust -d chrono

inl suffix = test_guid () |> sm'.obj_to_string |> sm'.range (am'.End fun x => x () - 6i32) (am'.End eval)
min_value ()
|> specify_date_kind Local
|> date_time_guid_from_date_time (test_guid ())
|> sm'.obj_to_string
|> fun s => s |> _assert_eq' $'$"00010101-0000-0000-0000-0{!(s |> sm'.slice 25i32 29)}{!suffix}"'

.fsx (fsharp)
__assert_eq' { expected = 00010101-0000-0000-0000-000000cba987 }
.rs (rust)
__assert_eq' { expected = 00010101-0000-0000-0000-000000cba987 }

In [22]:
///- --test
///> fsharp
///> rust -d chrono

inl suffix = test_guid () |> sm'.obj_to_string |> sm'.range (am'.End fun x => x () - 6i32) (am'.End eval)
max_value ()
|> specify_date_kind Utc
|> add_days -1
|> date_time_guid_from_date_time (test_guid ())
|> sm'.obj_to_string
|> fun s => s |> _assert_eq ("99991230-2359-5999-9999-9" ++# (s |> sm'.slice 25i32 29) ++# suffix)

.fsx (fsharp)
__assert_eq { expected = 99991230-2359-5999-9999-900000cba987 }
.rs (rust)
__assert_eq { expected = 99991230-2359-5999-9999-900000cba987 }

In [23]:
///- --test
///> fsharp
///> rust -d chrono

inl suffix = test_guid () |> sm'.obj_to_string |> sm'.range (am'.End fun x => x () - 6i32) (am'.End eval)
unix_epoch ()
|> specify_date_kind Utc
|> add_days 1
|> date_time_guid_from_date_time (test_guid ())
|> sm'.obj_to_string
|> fun s => s |> _assert_eq ("19700102-0000-0000-0000-0" ++# (s |> sm'.slice 25i32 29) ++# suffix)

.fsx (fsharp)
__assert_eq { expected = 19700102-0000-0000-0000-000000cba987 }
.rs (rust)
__assert_eq { expected = 19700102-0000-0000-0000-000000cba987 }

### date_time_from_guid

In [ ]:
inl date_time_from_guid (date_time_guid : date_time_guid) =
    inl date_time_guid = date_time_guid |> sm'.obj_to_string
    inl sm_replace = sm'.replace "-" ""
    run_target_args (fun () => sm_replace) function
        | Rust => fun _ =>
            $'{ let s = !date_time_guid[..25].replace("-", ""); chrono::NaiveDateTime::parse_from_str(&s[0..14], "%Y%m%d%H%M%S").unwrap() + chrono::TimeDelta::nanoseconds(s[14..21].parse::<i64>().unwrap() * 100) }' : date_time
        | TypeScript _ => fun sm_replace =>
            $'System.DateTime.Parse (!date_time_guid.[..24] |> !sm_replace)' : date_time
        | _ => fun sm_replace => $'System.DateTime.ParseExact (!date_time_guid.[..24] |> !sm_replace, "yyyyMMddHHmmssfffffff", null)' : date_time

In [24]:
///- --test
///> fsharp

date_time_from_guid (guid.new_guid "00010101-0000-0000-0000-0a9876543210")
|> _assert_eq' (min_value ())

__assert_eq' { expected = 01/01/0001 00:00:00 }

In [25]:
///- --test
///> fsharp

date_time_from_guid (guid.new_guid $'$"99991231-2359-5999-9999-9{(!test_guid () |> string |> fun s -> s.Substring (s.Length - 11))}"')
|> _assert_eq' (max_value ())

__assert_eq' { expected = 12/31/9999 23:59:59 }

In [26]:
///- --test
///> fsharp

date_time_from_guid (guid.new_guid $'$"19700101-0000-0000-0000-0{(!test_guid () |> string |> fun s -> s.Substring (s.Length - 11))}"')
|> _assert_eq' $'System.DateTime.UnixEpoch'

__assert_eq' { expected = 01/01/1970 00:00:00 }

### timestamp_guid_from_timestamp

In [ ]:
inl timestamp_guid_from_timestamp (guid : guid.guid) (timestamp : timestamp) : timestamp_guid =
    inl guid = guid |> sm'.obj_to_string
    inl timestamp = timestamp |> sm'.obj_to_string |> sm'.pad_left 18i32 '0'
    if is_rust ()
    then $'Rc::<str>::from(format\!("{}-{}-{}-{}{}", &!timestamp[0..8], &!timestamp[8..12], &!timestamp[12..16], &!timestamp[16..18], &!guid[21..]))'
    else $'`timestamp_guid $"{!timestamp.[0..7]}-{!timestamp.[8..11]}-{!timestamp.[12..15]}-{!timestamp.[16..17]}{!guid.[21..]}"'

In [27]:
///- --test
///> rust

timestamp_guid_from_timestamp (test_guid ()) (0i64 |> timestamp)
|> _assert_eq' (guid.new_guid "00000000-0000-0000-0043-210fedcba987")

__assert_eq' { expected = "00000000-0000-0000-0043-210fedcba987" }

0

In [28]:
///- --test
///> fsharp

timestamp_guid_from_timestamp (test_guid ()) (999999999999999999i64 |> timestamp)
|> _assert_eq' (guid.new_guid $'$"99999999-9999-9999-9943-2{(!test_guid () |> string |> fun s -> s.Substring (s.Length - 11))}"')

__assert_eq' { expected = 99999999-9999-9999-9943-210fedcba987 }

### timestamp_from_guid

In [ ]:
inl timestamp_from_guid (guid : date_time_guid) : timestamp =
    inl guid = guid |> sm'.obj_to_string
    if is_rust ()
    then $'format\!("{}{}{}{}", &!guid[0..8], &!guid[9..13], &!guid[14..18], &!guid[19..21]).parse::<i64>().unwrap()'
    else $'`i64 $"{!guid.[0..7]}{!guid.[9..12]}{!guid.[14..17]}{!guid.[19..20]}"'

In [29]:
///- --test
///> rust

timestamp_from_guid (guid.new_guid "00000000-0000-0000-00dc-ba9876543210")
|> _assert_eq' (0i64 |> timestamp)

__assert_eq' { expected = 0 }

0

In [30]:
///- --test
///> fsharp

timestamp_from_guid (guid.new_guid $'$"99999999-9999-9999-99{(!test_guid () |> string |> fun s -> s.Substring (s.Length - 15))}"')
|> _assert_eq' (999999999999999999i64 |> timestamp)

__assert_eq' { expected = 999999999999999999 }

### new_guid_from_date_time

In [ ]:
inl new_guid_from_date_time (date_time : date_time) =
    inl guid = guid.new_raw_guid ()
    date_time |> date_time_guid_from_date_time guid

In [31]:
///- --test
///> fsharp

utc_now ()
|> new_guid_from_date_time
|> date_time_from_guid
|> fun date_time => new_time_span date_time (utc_now ()) |> total_seconds |> convert.i32
|> _assert_eq 0

__assert_eq { expected = 0 }

### new_guid_from_timestamp

In [ ]:
inl new_guid_from_timestamp (timestamp : timestamp) =
    inl guid = guid.new_raw_guid ()
    timestamp |> timestamp_guid_from_timestamp guid

In [32]:
///- --test
///> fsharp

utc_now ()
|> to_timestamp
|> new_guid_from_timestamp
|> timestamp_from_guid
|> fun (timestamp t) =>
    (convert.convert t - (utc_now () |> to_timestamp |> fun (timestamp x) => convert.convert x)) / 100000i64
|> _assert_eq 0i64

__assert_eq { expected = 0 }

## main

In [33]:
///> fsharp

inl main () =
    $'let date_time_guid_from_date_time x = !date_time_guid_from_date_time x' : ()
    $'let date_time_from_guid x = !date_time_from_guid x' : ()
    $'let timestamp_guid_from_timestamp x = !timestamp_guid_from_timestamp x' : ()
    $'let timestamp_from_guid x = !timestamp_from_guid x' : ()
    $'let new_guid_from_date_time x = !new_guid_from_date_time x' : ()
    $'let new_guid_from_timestamp x = !new_guid_from_timestamp x' : ()
    $'let format x = !format x' : ()
    $'let format_iso8601 x = !format_iso8601 x' : ()